## Token Intervention: Prompt-Level Probability

Splices text from the source prompt into the base prompt at a chosen location (no model-internals patching) and measures how much this shifts the model's probability of answering with the factual sum vs. the counterfactual sum.

In [ ]:
%load_ext autoreload
%autoreload 2

### Set-up

In [ ]:
import sys
sys.path.append("src")

import torch
import gc
from tqdm import tqdm

import _config
from _intervention import get_label_probability

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS", # GPT-OSS or R1
    prompt_type="h_pre_result", # empty or pre_result or pre_sum
)
intervention_config = _config.InterventionConfig(
    intervention_loc="user_question_and_restatement",
    intervention_ids=[6, 8, 10, 12],
)
run_config = _config.RunConfig(
    experiment_root="experiments/token_intervention",
    result_dir="h_result_unfaithful_source",
    output_filename=f"probability{prompt_config.suffix}_{intervention_config.intervention_loc}.csv",
)
batch_size = 24

model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 divided prompts


In [9]:
intervention_ids = list(intervention_config.intervention_ids)
print(intervention_ids)

[6, 8, 10, 12]


### Run: prompt-level intervention + answer probability

For each batch, builds the intervened prompt and teacher-forces the factual and counterfactual final-answer probabilities.

In [ ]:
# Get header of prompts dataset
header = list(prompts.columns) + ['intervention_prompt', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]

    # Prepare batch of intervention prompts
    intervention_prompts = _config.build_intervened_prompts(batch_rows, intervention_ids)
    tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    
    base_labels, base_labels_mask = _config.prepare_sequence_label_tokens(tokenizer, batch_rows['base_sum'], device=model.device)
    base_labels_probs = get_label_probability(model, tokens, base_labels, base_labels_mask)

    source_labels, source_labels_mask = _config.prepare_sequence_label_tokens(tokenizer, batch_rows['source_sum'], device=model.device)
    source_labels_probs = get_label_probability(model, tokens, source_labels, source_labels_mask)
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        _config.write_to_csv(filepath, row.to_list() + [intervention_prompts[j], base_labels_probs[j].item(), source_labels_probs[j].item()])


  0%|                                                                             | 0/11 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████| 11/11 [02:07<00:00, 11.58s/it]
